In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Load dataset
data = pd.read_csv('/content/drive/MyDrive/HotelOccupancyForecastingBookingAndCancellation/data/raw/data.csv')

# Strip trailing spaces from column names
data.columns = data.columns.str.strip()

# Handle missing values
data = data.dropna()  # or use other imputation methods

# Handle duplicates
data = data.drop_duplicates()

# Strip leading and trailing spaces from date columns
data['check_in'] = data['check_in'].str.strip()
data['check_out'] = data['check_out'].str.strip()

# Convert date columns to datetime
data['check_in'] = pd.to_datetime(data['check_in'], format='%d/%m/%Y', dayfirst=True)
data['check_out'] = pd.to_datetime(data['check_out'], format='%d/%m/%Y', dayfirst=True)

# Extract features from dates
data['check_in_year'] = data['check_in'].dt.year
data['check_in_month'] = data['check_in'].dt.month
data['check_in_day'] = data['check_in'].dt.day
data['check_in_dayofweek'] = data['check_in'].dt.dayofweek
data['check_in_is_weekend'] = data['check_in'].dt.dayofweek >= 5
data['check_out_year'] = data['check_out'].dt.year
data['check_out_month'] = data['check_out'].dt.month
data['check_out_day'] = data['check_out'].dt.day
data['check_out_dayofweek'] = data['check_out'].dt.dayofweek
data['check_out_is_weekend'] = data['check_out'].dt.dayofweek >= 5
data['days'] = (data['check_out'] - data['check_in']).dt.days
data['month'] = data['check_in'].dt.month
data['year'] = data['check_in'].dt.year

# Drop original date columns
data = data.drop(columns=['check_in', 'check_out'])

# Scale the data
scaler = StandardScaler()
scaled_features = scaler.fit_transform(data[[
    'check_in_year', 'check_in_month', 'check_in_day', 'check_in_dayofweek', 'check_in_is_weekend',
    'check_out_year', 'check_out_month', 'check_out_day', 'check_out_dayofweek', 'check_out_is_weekend',
    'days', 'month', 'year'
]])
scaled_data = pd.DataFrame(scaled_features, columns=[
    'check_in_year', 'check_in_month', 'check_in_day', 'check_in_dayofweek', 'check_in_is_weekend',
    'check_out_year', 'check_out_month', 'check_out_day', 'check_out_dayofweek', 'check_out_is_weekend',
    'days', 'month', 'year'
])

# Add the target column back to the scaled data
scaled_data['is_cancelled'] = data['is_cancelled']

# Save processed data
scaled_data.to_csv('/content/drive/MyDrive/HotelOccupancyForecastingBookingAndCancellation/data/processed/processed_scaled_dataset.csv', index=False)

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import joblib

# Load processed data
data = pd.read_csv('/content/drive/MyDrive/HotelOccupancyForecastingBookingAndCancellation/data/processed/processed_scaled_dataset.csv')

# Define essential features and target variable
essential_features = [
    'check_in_year', 'check_in_month', 'check_in_day', 'check_in_dayofweek', 'check_in_is_weekend',
    'check_out_year', 'check_out_month', 'check_out_day', 'check_out_dayofweek', 'check_out_is_weekend',
    'days', 'month', 'year'
]
X = data[essential_features]
y_cancellation = data['is_cancelled']

# Define target variable for bookings
y_booking = 1 - data['is_cancelled']  # Booking is the opposite of cancellation

# Save the essential feature names and their order
with open('/content/drive/MyDrive/HotelOccupancyForecastingBookingAndCancellation/models/essential_feature_names.txt', 'w') as f:
    for item in essential_features:
        f.write("%s\n" % item)

# Split the data for cancellations
X_train, X_test, y_train_cancel, y_test_cancel = train_test_split(X, y_cancellation, test_size=0.2, random_state=42)

# Split the data for bookings
_, _, y_train_book, y_test_book = train_test_split(X, y_booking, test_size=0.2, random_state=42)

# Hyperparameter tuning and model training function
def train_model(model, param_grid, X_train, y_train):
    grid_search = GridSearchCV(model, param_grid, cv=5, scoring='accuracy')
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    return best_model

# Extensive hyperparameter tuning for Random Forest
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
rf_model_cancel = train_model(RandomForestClassifier(), rf_param_grid, X_train, y_train_cancel)
joblib.dump(rf_model_cancel, '/content/drive/MyDrive/HotelOccupancyForecastingBookingAndCancellation/models/rf_model_cancel.pkl')

lr_param_grid = {'C': [0.01, 0.1, 1, 10]}
lr_model_cancel = train_model(LogisticRegression(penalty='l2', max_iter=1000), lr_param_grid, X_train, y_train_cancel)
joblib.dump(lr_model_cancel, '/content/drive/MyDrive/HotelOccupancyForecastingBookingAndCancellation/models/lr_model_cancel.pkl')

xgb_param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 6, 9],
    'subsample': [0.6, 0.8, 1.0]
}
xgb_model_cancel = train_model(xgb.XGBClassifier(), xgb_param_grid, X_train, y_train_cancel)
joblib.dump(xgb_model_cancel, '/content/drive/MyDrive/HotelOccupancyForecastingBookingAndCancellation/models/xgb_model_cancel.pkl')

# Train models for bookings
rf_model_book = train_model(RandomForestClassifier(), rf_param_grid, X_train, y_train_book)
joblib.dump(rf_model_book, '/content/drive/MyDrive/HotelOccupancyForecastingBookingAndCancellation/models/rf_model_book.pkl')

lr_model_book = train_model(LogisticRegression(penalty='l2', max_iter=1000), lr_param_grid, X_train, y_train_book)
joblib.dump(lr_model_book, '/content/drive/MyDrive/HotelOccupancyForecastingBookingAndCancellation/models/lr_model_book.pkl')

xgb_model_book = train_model(xgb.XGBClassifier(), xgb_param_grid, X_train, y_train_book)
joblib.dump(xgb_model_book, '/content/drive/MyDrive/HotelOccupancyForecastingBookingAndCancellation/models/xgb_model_book.pkl')

['/content/drive/MyDrive/HotelOccupancyForecastingBookingAndCancellation/models/xgb_model_book.pkl']

In [6]:
import pandas as pd
import joblib
from datetime import datetime
from sklearn.preprocessing import StandardScaler

# Load the saved essential feature names
with open('/content/drive/MyDrive/HotelOccupancyForecastingBookingAndCancellation/models/essential_feature_names.txt', 'r') as f:
    essential_features = f.read().splitlines()

# Function to extract date features
def extract_date_features(check_in_date, check_out_date):
    # Convert dates to datetime
    check_in = pd.to_datetime(check_in_date)
    check_out = pd.to_datetime(check_out_date)

    # Create a DataFrame with all dates in the range
    date_range = pd.date_range(start=check_in, end=check_out, freq='D')

    features_list = []
    for date in date_range:
        features_list.append({
            'check_in_year': date.year,
            'check_in_month': date.month,
            'check_in_day': date.day,
            'check_in_dayofweek': date.dayofweek,
            'check_in_is_weekend': date.dayofweek >= 5,
            'check_out_year': date.year,
            'check_out_month': date.month,
            'check_out_day': date.day,
            'check_out_dayofweek': date.dayofweek,
            'check_out_is_weekend': date.dayofweek >= 5,
            'days': (check_out - check_in).days,
            'month': date.month,
            'year': date.year
        })

    return pd.DataFrame(features_list)

# Sample check-in and check-out dates
check_in_date = '1/01/2026'
check_out_date = '1/01/2029'

# Extract features from input dates
input_df = extract_date_features(check_in_date, check_out_date)

# Scale the input data
scaler = StandardScaler()
scaled_input_df = pd.DataFrame(scaler.fit_transform(input_df), columns=input_df.columns)

# Ensure the columns are in the same order as during training
scaled_input_df = scaled_input_df[essential_features]

# Load trained model for cancellations
rf_model_cancel = joblib.load('/content/drive/MyDrive/HotelOccupancyForecastingBookingAndCancellation/models/rf_model_cancel.pkl')

# Predict cancellation
scaled_input_df['predicted_cancellation'] = rf_model_cancel.predict(scaled_input_df)

# Debug: Check if the column is added correctly
print(scaled_input_df.head())

# Remove unnecessary columns for booking prediction
booking_input_df = scaled_input_df[essential_features]

# Load trained model for bookings
rf_model_book = joblib.load('/content/drive/MyDrive/HotelOccupancyForecastingBookingAndCancellation/models/rf_model_book.pkl')

# Predict booking
booking_input_df['predicted_booking'] = rf_model_book.predict(booking_input_df)

# Add year, month, and day columns to booking_input_df
booking_input_df['year'] = input_df['check_in_year']
booking_input_df['month'] = input_df['check_in_month']
booking_input_df['day'] = input_df['check_in_day']

# Add year, month, and day columns to scaled_input_df (for cancellation results)
scaled_input_df['year'] = input_df['check_in_year']
scaled_input_df['month'] = input_df['check_in_month']
scaled_input_df['day'] = input_df['check_in_day']

# Summarize the results for different time frames
def summarize_results(input_df, prediction_column):
    yearly = input_df.groupby('year')[prediction_column].sum()
    monthly = input_df.groupby(['year', 'month'])[prediction_column].sum()
    daily = input_df.groupby(['year', 'month', 'day'])[prediction_column].sum()
    per_month = input_df.groupby('month')[prediction_column].sum()
    return yearly, monthly, daily, per_month

# Summarize cancellation results using scaled_input_df with the predicted_cancellation column
yearly_cancellations, monthly_cancellations, daily_cancellations, cancellations_per_month = summarize_results(scaled_input_df, 'predicted_cancellation')

# Summarize booking results using booking_input_df with the predicted_booking column
yearly_bookings, monthly_bookings, daily_bookings, bookings_per_month = summarize_results(booking_input_df, 'predicted_booking')

# Display results
print("Yearly Cancellations:")
print(yearly_cancellations)
print("\nMonthly Cancellations:")
print(monthly_cancellations)
print("\nDaily Cancellations:")
print(daily_cancellations)
print("\nCancellations per Month:")
print(cancellations_per_month)

print("\nYearly Bookings:")
print(yearly_bookings)
print("\nMonthly Bookings:")
print(monthly_bookings)
print("\nDaily Bookings:")
print(daily_bookings)
print("\nBookings per Month:")
print(bookings_per_month)


   check_in_year  check_in_month  check_in_day  check_in_dayofweek  \
0      -1.225034       -1.598424     -1.671010           -0.001367   
1      -1.225034       -1.598424     -1.557484            0.498463   
2      -1.225034       -1.598424     -1.443959            0.998292   
3      -1.225034       -1.598424     -1.330433            1.498122   
4      -1.225034       -1.598424     -1.216908           -1.500856   

   check_in_is_weekend  check_out_year  check_out_month  check_out_day  \
0            -0.633263       -1.225034        -1.598424      -1.671010   
1            -0.633263       -1.225034        -1.598424      -1.557484   
2             1.579123       -1.225034        -1.598424      -1.443959   
3             1.579123       -1.225034        -1.598424      -1.330433   
4            -0.633263       -1.225034        -1.598424      -1.216908   

   check_out_dayofweek  check_out_is_weekend  days     month      year  \
0            -0.001367             -0.633263   0.0 -1.598424